# Querying IDL Metadata and Downloading PDFs from AWS Open Data

This notebook demonstrates a workflow for batch downloading documents from the Industry Documents Library (IDL).

The process:
1. **Query** `metadata.idl.ucsf.edu` to get document IDs by industry using cursor-based pagination
2. **Build** AWS S3 URLs for each document
3. **Download** PDFs in batches to the local `downloads/` directory
4. **Save** metadata to a CSV file as you go (memory-efficient, batch-by-batch)

**Note:** Documents are processed in batches to keep memory usage constant regardless of dataset size.

## Setup

Install dependencies for querying Solr, downloading files, and data processing.

In [40]:
# Example dependencies
!pip install requests pandas tqdm

import requests
import pandas as pd
from pathlib import Path



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## Configure Industry

Choose which industry you want to download documents for.

Available industries: `tobacco`, `opioids`, `chemical`, `drug`, `food`, `fossilfuel`

Edit the `industry` variable below to change your selection.

In [41]:
# Example: user selects a single industry
# available industries:
#   tobacco
#   opioids
#   chemical
#   drug
#   food
#   fossilfuel
#  update the industry the one you are interested in:
industry = "tobacco"

print(f"Selected industry: {industry}")


Selected industry: tobacco


## Solr Query Function

This function queries a single batch of documents using cursor-based pagination.
Cursor marks ensure stable pagination through large result sets.

In [42]:
SOLR_ENDPOINT = "https://metadata.idl.ucsf.edu/solr/ltdl3/query"

def query_ids_for_industry(industry, cursor_mark="*", rows=100):
    """Query a single batch of documents for an industry.
    
    Returns a tuple of (docs, next_cursor_mark)
    """
    params = {
        "q": f"industry:{industry}",
        "rows": rows,
        "wt": "json",
        "cursorMark": cursor_mark,
        "sort": "id asc",
    }

    response = requests.get(SOLR_ENDPOINT, params=params)
    print(f"Response status: {response.status_code}")
    print(f"Response headers: {response.headers}")
    print(f"Response text (first 1000 chars): {response.text[:1000]}")
    response.raise_for_status()
    
    data = response.json()
    
    docs = data["response"]["docs"]
    next_cursor_mark = data["nextCursorMark"]
    
    return docs, next_cursor_mark

## Fetch, Build URLs, and Download Documents

Iterates through all documents for the selected industry in batches:
- Queries Solr for a batch of IDs
- Builds S3 URLs for each document
- Downloads PDFs to `downloads/` directory
- Saves metadata to CSV (appends each batch)

This batch-by-batch approach keeps memory usage constant.

In [43]:
def build_pdf_url(doc_id):
    AWS_BASE_URL = "https://ucsf-idl-dataset.s3.us-east-1.amazonaws.com"
    return f"{AWS_BASE_URL}/{doc_id[0]}/{doc_id[1]}/{doc_id[2]}/{doc_id[3]}/{doc_id}/{doc_id}.pdf"

In [44]:

def download_pdf(doc_id, pdf_url):
    download_dir = Path("downloads")
    download_dir.mkdir(exist_ok=True)
    output_path = download_dir / f"{doc_id}.pdf"

    response = requests.get(pdf_url, stream=True)

    if response.status_code == 200:
        with open(output_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Downloaded: {output_path}")
    else:
        print(f"Failed: {doc_id} ({response.status_code})")

In [45]:
# Paginate through all documents and download (batched processing)
cursor_mark = "*"
batch_num = 0
total_processed = 0
csv_file = "idl_metadata_export.csv"

# Remove file if it exists to start fresh
import os
if os.path.exists(csv_file):
    os.remove(csv_file)

while True:
    try:
        docs, next_cursor_mark = query_ids_for_industry(industry, cursor_mark)
    except Exception as e:
        print(f"Error fetching batch: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        break

    if not docs:
        print("No more documents to fetch")
        break

    batch_num += 1
    print(f"\n--- Batch {batch_num} ({len(docs)} documents) ---")

    # Process batch
    batch_data = []
    for doc in docs:
        doc["industry"] = industry
        batch_data.append(doc)
        
        doc_id = doc["id"]
        pdf_url = build_pdf_url(doc_id)
        download_pdf(doc_id, pdf_url)

    # optional: Save batch to CSV (append mode)
    batch_df = pd.DataFrame(batch_data)
    batch_df.to_csv(csv_file, mode='a', header=(batch_num == 1), index=False)
    
    total_processed += len(docs)
    print(f"Saved batch {batch_num} to CSV. Total processed: {total_processed}")

    if next_cursor_mark == cursor_mark:
        print("Reached end of results")
        break

    cursor_mark = next_cursor_mark

print(f"\nCompleted. Total documents processed: {total_processed}")

Response status: 200
Response headers: {'Date': 'Sat, 23 May 2026 04:56:18 GMT', 'Content-Type': 'text/plain;charset=utf-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Server': 'Apache', 'Strict-Transport-Security': 'max-age=63072000; includeSubDomains', 'Content-Security-Policy': "default-src 'none'; base-uri 'none'; connect-src 'self'; form-action 'self'; font-src 'self'; frame-ancestors 'none'; img-src 'self'; media-src 'self'; style-src 'self' 'unsafe-inline'; script-src 'self'; worker-src 'self';", 'X-Content-Type-Options': 'nosniff', 'X-Frame-Options': 'SAMEORIGIN', 'X-XSS-Protection': '1; mode=block', 'Vary': 'Accept-Encoding,User-Agent', 'Content-Encoding': 'gzip', 'Access-Control-Allow-Origin': '*'}
Response text (first 1000 chars): {
  "responseHeader":{
    "status":0,
    "QTime":325,
    "params":{
      "q":"industry:tobacco",
      "cursorMark":"*",
      "sort":"id asc",
      "rows":"100",
      "wt":"json"}},
  "response":{"numFound":19956191,"start"

KeyboardInterrupt: 

# What is one question that you have answered using these data? Can you show us how you came to that answer?

One question that has been answered using these data is: How did Juul and related e-cigarette companies use product placement and external partnerships to market their products and increase their credibility?

Researchers answered this question by analyzing internal documents from companies such as Ploom, Pax Labs, and Juul that were available through the Industry Documents Library (IDL). These documents included communications, reports, and other records that provided insight into the companies' marketing strategies. Through this analysis, researchers were able to reconstruct product-placement efforts in music videos, television programs, and films, as well as examine relationships with scientific experts and public-health influencers.

The process involved collecting and organizing large numbers of documents and metadata from multiple sources. As described in the publication, researchers used a combination of manual review and computational methods, including R, to gather data, extract relevant information, and map relationships among organizations, authors, funders, and publishers. By systematically analyzing these connections, they were able to identify patterns and networks that would have been difficult to observe through individual document searches alone.

This example demonstrates how the IDL can be used not only to locate documents but also to conduct large-scale analyses that reveal corporate strategies, professional relationships, and patterns of influence.

# What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

One unanswered research question that could be explored using these data is: How many customer complaints did Juul receive in 2018, and what trends can be identified from those complaints over time? Questions like this cannot easily be answered through the standard IDL website interface, but they become possible when the archive is treated as a dataset for analysis.

Another valuable research question would be to examine relationships among key actors within and across industries. By accessing the underlying data, researchers could generate network graphs that reveal connections between companies, public relations firms, lobbying organizations, and purported grassroots advocacy groups. These analyses could provide insights into how influence and communication networks are structured.

For researchers interested in answering these types of questions, I would recommend working directly with the archived data rather than relying solely on the website interface. Open access to the data allows for bulk downloading and organization of documents in ways that better support research objectives. For example, a researcher could download and analyze all available Juul Slack messages, group conversations by topic or time period, and apply quantitative or qualitative methods to identify patterns that would be difficult to detect through the IDL website alone.
